<div style="background:linear-gradient(135deg,#1B3A1B,#2E5E2E,#3A7A3A);padding:36px 28px;border-radius:14px;color:#fff;text-align:center">
<h1 style="margin:0;font-size:2.2em">⚙️ Notebook 5 — Reactor de Lecho Empacado (PBR)</h1>
<h2 style="margin:10px 0 4px;font-weight:300;opacity:.92">Caída de Presión (Ergun) · Cinética Catalítica · Desactivación · Diseño Industrial</h2>
<p style="margin:4px 0 0;opacity:.85">Fogler Capítulos 5, 10 y 12 · 740484 Diseño de Reactores · Universidad del Valle</p>
<p style="margin:4px 0 0;opacity:.75;font-size:.88em;font-style:italic">Prof. José Antonio Lara Ramos, Ing.Qco., M.Sc., Ph.D.</p>
</div>

## 🎯 Objetivos del módulo

1. **Derivar** la ecuación de diseño del PBR en términos de masa de catalizador $W$ y comparar con el PFR volumétrico.
2. **Integrar** el sistema de ODEs acopladas: balance de masa + ecuación de Ergun (caída de presión) + balance de energía.
3. **Modelar** la cinética de Langmuir–Hinshelwood–Hougen–Watson (LHHW) para reacciones catalíticas.
4. **Simular** la desactivación del catalizador y calcular el tiempo de ciclo de regeneración.
5. **Diseñar** cuantitativamente un PBR industrial con cálculo de masa de catalizador, caída de presión y perfil de conversión.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import odeint, solve_ivp
from scipy.optimize import brentq
import ipywidgets as widgets
from ipywidgets import interact, interactive
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0A0F1E','axes.facecolor':'#0A1530',
    'axes.edgecolor':'#2A4080','grid.color':'#1A2840','grid.alpha':.4,
    'text.color':'#C8D8F0','axes.labelcolor':'#A0C0FF',
    'xtick.color':'#8090B0','ytick.color':'#8090B0',
    'font.size':11,'axes.titlesize':12,'axes.titlepad':10,
    'lines.linewidth':2.2,'figure.dpi':110
})
print("✅ Librerías cargadas correctamente")


---
# 1. Ecuación de diseño del PBR — derivación rigurosa (Fogler §10.1)

## 1.1 ¿Por qué masa de catalizador W en lugar de volumen V?

En el PFR homogéneo, la variable independiente es el **volumen** $V$ [dm³].  
En el PBR (**Packed Bed Reactor**), la reacción ocurre en la **superficie del catalizador sólido**,  
por lo que la velocidad de reacción se expresa por unidad de **masa de catalizador** $W$ [kg]:

$$-r'_A \quad\left[\frac{\text{mol}}{\text{kg}_\text{cat}\cdot\text{s}}\right] \quad\neq\quad -r_A \quad\left[\frac{\text{mol}}{\text{dm}^3\cdot\text{s}}\right]$$

La relación entre ambas es: $\quad -r_A = \rho_b\,(-r'_A)$  
donde $\rho_b$ [kg/dm³] es la densidad aparente del lecho empacado.

## 1.2 Derivación diferencial (Fogler §10.1, Ec. 10-1)

Aplicando el balance molar a un elemento diferencial de masa de catalizador $dW$:

$$F_{A0}\,\frac{dX}{dW} = -r'_A(X,T,P) \tag{10-1}$$

Integración:

$$\boxed{W = F_{A0}\int_0^{X_A}\frac{dX}{-r'_A(X,T,P)}}$$

## 1.3 Sistema completo de ODEs para el PBR no isotérmico con caída de presión

$$\frac{dX}{dW} = \frac{-r'_A(X,T,P)}{F_{A0}} \tag{I — masa}$$

$$\frac{dT}{dW} = \frac{Ua(T_a-T) + (-r'_A)(-\Delta H_{rx})}{\dot{m}\,C_p} \tag{II — energía}$$

$$\frac{dP}{dW} = -\frac{\alpha}{2(1-\alpha W/W_{tot})} \cdot \frac{T}{T_0} \cdot \frac{F_T}{F_{T0}} \tag{III — Ergun simplificada}$$

o equivalentemente en términos de la variable de presión adimensional $y = P/P_0$:

$$\frac{dy}{dW} = -\frac{\alpha}{2y}\cdot\frac{F_T/F_{T0}}{T_0/T} \tag{III bis}$$


---
# 2. Ecuación de Ergun — caída de presión en lecho empacado (Fogler §5.4)

La ecuación de Ergun relaciona la caída de presión con las propiedades del lecho y del fluido:

$$-\frac{dP}{dz} = \frac{G}{\rho\,D_p}\cdot\frac{1-\phi}{\phi^3}\left[\frac{150(1-\phi)\mu}{D_p} + 1.75\,G\right]$$

donde:
| Símbolo | Definición | Unidades |
|---------|-----------|---------|
| $G = \rho\,u_0$ | flujo másico superficial | kg/(m²·s) |
| $D_p$ | diámetro equivalente de partícula | m |
| $\phi$ | porosidad del lecho (void fraction) | adimensional |
| $\mu$ | viscosidad dinámica del fluido | Pa·s |
| $\rho$ | densidad del fluido | kg/m³ |

## 2.1 Forma compacta de Fogler para el PBR (Ec. 5-28)

$$\frac{dy}{dW} = -\frac{\alpha}{2y}\cdot\frac{T}{T_0}\cdot\frac{F_T}{F_{T0}}$$

donde el **parámetro de Ergun** $\alpha$ [kg⁻¹] es:

$$\alpha = \frac{2\beta_0}{A_c\,\rho_c(1-\phi)\,P_0} \quad\text{con}\quad \beta_0 = \frac{G\,(1-\phi)}{D_p\,\phi^3\,\rho_0}\left[\frac{150(1-\phi)\mu}{D_p}+1.75\,G\right]$$

## 2.2 Implicación de diseño

Para reacción con cambio de número de moles ($\delta \neq 0$) y caída de presión:

$$C_A = C_{A0}\cdot\frac{1-X}{1+\varepsilon X}\cdot\frac{P}{P_0}\cdot\frac{T_0}{T} = C_{A0}\cdot\frac{1-X}{1+\varepsilon X}\cdot y\cdot\frac{T_0}{T}$$

La caída de presión **reduce** la concentración de reactivos → **reduce** la velocidad de reacción → se necesita **más catalizador** para alcanzar la misma conversión.


---
# 3. Cinética catalítica — Modelo de Langmuir–Hinshelwood–Hougen–Watson (LHHW)

## 3.1 Mecanismo de superficie

Para la reacción $A + B \to C + D$ sobre un catalizador sólido:

| Paso | Proceso | Ecuación |
|------|---------|---------|
| 1 | Adsorción de A en sitio libre | $A + S \rightleftharpoons A\cdot S$ |
| 2 | Adsorción de B en sitio libre | $B + S \rightleftharpoons B\cdot S$ |
| 3 | Reacción superficial (paso controlante) | $A\cdot S + B\cdot S \rightleftharpoons C\cdot S + D\cdot S$ |
| 4 | Desorción de C | $C\cdot S \rightleftharpoons C + S$ |
| 5 | Desorción de D | $D\cdot S \rightleftharpoons D + S$ |

## 3.2 Expresión de velocidad LHHW (Fogler §10.3)

Si el **paso controlante es la reacción superficial** (los pasos de adsorción/desorción están en equilibrio):

$$\boxed{-r'_A = \frac{k\,C_A\,C_B}{(1 + K_A\,C_A + K_B\,C_B + K_C\,C_C + K_D\,C_D)^2}}$$

Estructura general:

$$-r'_A = \frac{[\text{cinética}] \times [\text{fuerza impulsora}]}{[\text{grupo de adsorción}]^n}$$

donde $n$ = número de sitios involucrados en la reacción superficial.

## 3.3 Casos especiales importantes

- **Michaelis-Menten** (enzimas): $-r'_A = V_\max C_A/(K_m + C_A)$ — LHHW con $K_A C_A \ll 1$, $n=1$  
- **Cinética de primer orden** (catalizador no saturado): $-r'_A = k'C_A$ — $K_j C_j \ll 1$ para todos  
- **Cinética de orden cero** (catalizador saturado): $-r'_A = k'$ — $K_A C_A \gg 1$


---
# 4. Ejemplo Industrial: Hidrodesulfuración catalítica (HDS)

**Contexto:** La hidrodesulfuración es uno de los procesos catalíticos más importantes en refinerías.
Elimina el azufre del gasóleo mediante:

$$\text{R-S} + \text{H}_2 \xrightarrow{\text{CoMo/Al}_2\text{O}_3} \text{R-H} + \text{H}_2\text{S}$$

**Parámetros de referencia** (proceso típico de refinería):

| Parámetro | Valor | Justificación |
|-----------|-------|--------------|
| $k'$ a 350°C | 0.12 dm³/(kg_cat·s) | Cinética pseudo-primer orden en S |
| $E_a$ | 105 kJ/mol | Reacción de hidrolisis C-S |
| $C_{A0}$ (azufre) | 0.25 mol/dm³ | ~8,000 ppm de azufre |
| $v_0$ | 15 dm³/s | Caudal de gasóleo |
| $X_{des}$ | 0.997 | Regulación EU: &lt;10 ppm S en diesel |
| $\alpha$ | 0.0028 kg⁻¹ | Ecuación de Ergun para lecho CoMo/Al₂O₃ |
| $P_0$ | 50 atm | Presión operativa |
| $T_0$ | 623 K (350°C) | Temperatura de entrada |


In [ ]:
# ═══════════════════════════════════════════════════
# PARÁMETROS BASE — PBR Hidrodesulfuración (HDS)
# ═══════════════════════════════════════════════════

# Cinéticos
k_ref  = 0.12          # dm³/(kg_cat·s) a T_ref
Ea     = 105e3         # J/mol
R_gas  = 8.314         # J/(mol·K)
T_ref  = 623.15        # K (350°C)
dHrx   = -63e3         # J/mol (exotérmica, hidrogenación)
n_ord  = 1.0           # pseudo-primer orden en azufre

# Alimentación
CA0    = 0.25          # mol/dm³ — 8000 ppm de S
v0     = 15.0          # dm³/s
FA0    = CA0 * v0      # mol/s
rho_cp = 950 * 1.8     # J/(dm³·K) — ρ·Cp fluido (gasóleo)
Ua_val = 0.0           # J/(kg·s·K) — adiabático para HDS

# Presión (Ergun)
alpha  = 0.0028        # kg⁻¹
P0     = 50.0          # atm
T0     = T_ref         # K
eps    = 0.0           # sin cambio de moles (H2 en exceso → δ≈0)

# Diseño
X_des  = 0.997         # conversión objetivo (EU &lt;10 ppm)
Ta     = 623.15        # K — temperatura fluido de calefacción

# ═══════════════════════════════════════════════════
# SISTEMA DE ODEs: dX/dW, dT/dW, dy/dW
# ═══════════════════════════════════════════════════
def pbr_odes(W, state, Ua=0.0, Ta=623.15, alpha=0.0028):
    X, T, y = state
    X = max(min(X, 0.9999), 0.0)
    y = max(y, 1e-4)

    k   = k_ref * np.exp(-Ea/R_gas * (1/T - 1/T_ref))
    P   = y * P0
    CA  = CA0 * (1-X) / (1+eps*X) * y * (T0/T)
    rA  = k * CA**n_ord                         # mol/(kg·s)

    dXdW = rA / FA0
    dTdW = (Ua*(Ta-T) + rA*(-dHrx)) / (FA0 * rho_cp / CA0)
    dydW = -alpha/(2*y) * (T0/T)                # sin cambio de moles

    return [dXdW, dTdW, dydW]

# Integración
W_span = np.linspace(0, 6000, 3000)          # kg de catalizador
sol = solve_ivp(
    pbr_odes, [0, 6000], [0.0, T0, 1.0],
    t_eval=W_span, method='RK45',
    rtol=1e-8, atol=1e-10,
    args=(Ua_val, Ta, alpha)
)

W_arr = sol.t
X_arr = sol.y[0]
T_arr = sol.y[1]
y_arr = sol.y[2]
P_arr = y_arr * P0
CA_arr = CA0*(1-X_arr)/(1+eps*X_arr)*y_arr*(T0/T_arr)

# Encontrar W necesario para X_des
try:
    from scipy.interpolate import interp1d
    interp_W = interp1d(X_arr, W_arr, kind='linear', fill_value='extrapolate')
    W_needed = float(interp_W(X_des))
except:
    W_needed = W_arr[np.searchsorted(X_arr, X_des)]

print(f"╔══════════════════════════════════════════════════╗")
print(f"║   RESULTADO DEL DISEÑO — PBR Hidrodesulfuración ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Conversión objetivo (X_des)    : {X_des:.3f}          ║")
print(f"║  Masa de catalizador necesaria  : {W_needed:.0f} kg     ║")
print(f"║  Caída de presión total         : {(1-y_arr[-1])*100:.1f} %     ║")
print(f"║  P a la salida                  : {P_arr[-1]:.1f} atm   ║")
print(f"║  T a la salida (adiabático)     : {T_arr[-1]-273.15:.1f} °C  ║")
print(f"║  ΔT adiabático (X_des)          : {T_arr[-1]-T0:.1f} K     ║")
print(f"╚══════════════════════════════════════════════════╝")


In [ ]:
# ═══════════════════════════════════════════════════
# PANEL DE RESULTADOS — 4 GRÁFICAS
# ═══════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('PBR Hidrodesulfuración — Perfiles a lo largo del lecho de catalizador',
             fontsize=14, color='#C8D8F0', fontweight='bold', y=1.01)

colors = {'X':'#FFD080','T':'#FF8C42','P':'#4A9EFF','CA':'#2ECC71'}

# -- Perfil X(W) --
ax = axes[0,0]
ax.plot(W_arr/1000, X_arr*100, color=colors['X'], lw=2.5, label='X (%)')
ax.axhline(X_des*100, color='#FF4444', ls='--', lw=1.5, label=f'X_des = {X_des*100:.1f}%')
if W_needed < W_arr[-1]:
    ax.axvline(W_needed/1000, color='#FF4444', ls=':', lw=1.5)
    ax.annotate(f'W = {W_needed/1000:.1f} t', xy=(W_needed/1000, X_des*100*0.85),
                color='#FF8080', fontsize=10)
ax.set_xlabel('Masa catalizador W (toneladas)'); ax.set_ylabel('Conversión X (%)')
ax.set_title('Perfil de Conversión X(W)'); ax.legend(loc='lower right'); ax.grid(True)

# -- Perfil T(W) --
ax = axes[0,1]
ax.plot(W_arr/1000, T_arr-273.15, color=colors['T'], lw=2.5)
ax.set_xlabel('W (toneladas)'); ax.set_ylabel('T (°C)')
ax.set_title('Perfil de Temperatura T(W)'); ax.grid(True)

# -- Perfil P(W) --
ax = axes[1,0]
ax.plot(W_arr/1000, y_arr, color=colors['P'], lw=2.5, label='y = P/P₀')
ax.fill_between(W_arr/1000, y_arr, 1, alpha=0.12, color=colors['P'])
ax.set_xlabel('W (toneladas)'); ax.set_ylabel('y = P/P₀ (adimensional)')
ax.set_title('Caída de Presión — Ecuación de Ergun'); ax.legend(); ax.grid(True)

# -- Perfil CA(W) --
ax = axes[1,1]
ax.plot(W_arr/1000, CA_arr*1000, color=colors['CA'], lw=2.5, label='C_A (mmol/dm³)')
ax.set_xlabel('W (toneladas)'); ax.set_ylabel('C_A (mmol/dm³)')
ax.set_title('Perfil de Concentración C_A(W)'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('pbr_perfiles.png', dpi=120, bbox_inches='tight',
            facecolor='#0A0F1E', edgecolor='none')
plt.show()
print("✅ Figura guardada: pbr_perfiles.png")


---
# 5. Desactivación del catalizador (Fogler §10.7)

## 5.1 Tipos de desactivación y modelos matemáticos

| Tipo | Causa | Cinética de desactivación |
|------|-------|--------------------------|
| **Envenenamiento** | Adsorción irreversible de impurezas (S, Pb, As) | $\dfrac{da}{dt} = -k_d\,C_\text{veneno}\,a^m$ |
| **Sinterización** | Coalescencia de cristalites metálicos a alta T | $\dfrac{da}{dt} = -k_d(T)\,a^n \quad (n=2)$ |
| **Deposición de coque** | Polimerización de intermedios sobre la superficie | $\dfrac{da}{dt} = -k_d\,p_A\,a^m$ |

donde $a(t)$ es la **actividad del catalizador** ($a=1$: fresco; $a=0$: desactivado).

## 5.2 Modelo PSSH para coque (Fogler §10.7.2)

La velocidad efectiva con desactivación:

$$-r'_A = a(t)\cdot(-r'_A)_\text{sin desactivación}$$

Para desactivación por coque (cinética de 1er orden en actividad):

$$\frac{da}{dt} = -k_d\,a \quad\Rightarrow\quad a(t) = e^{-k_d\,t}$$

**Tiempo de vida del catalizador** (criterio $a = a_\min$):

$$t_\text{vida} = \frac{-\ln(a_\min)}{k_d}$$


In [ ]:
# ═══════════════════════════════════════════════════
# MODELO DE DESACTIVACIÓN DEL CATALIZADOR PBR
# ═══════════════════════════════════════════════════
kd_vals = [1e-4, 5e-4, 2e-3]   # s⁻¹ — diferentes tasas de desactivación
a_min   = 0.20                   # actividad mínima aceptable (20%)
W_fix   = W_needed               # kg de catalizador del diseño anterior

t_vida_label = []
t_days = np.linspace(0, 60, 500)   # 0 a 60 días
t_s    = t_days * 86400            # en segundos

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo — actividad vs tiempo
for kd in kd_vals:
    a_t = np.exp(-kd * t_s)
    tv  = -np.log(a_min) / kd / 86400
    label = f'$k_d$ = {kd:.0e} s⁻¹  →  t_vida = {tv:.1f} d'
    t_vida_label.append((kd, tv))
    ax1.plot(t_days, a_t, lw=2.5, label=label)

ax1.axhline(a_min, color='#FF4444', ls='--', lw=1.5, label=f'a_min = {a_min}')
ax1.set_xlabel('Tiempo de operación (días)')
ax1.set_ylabel('Actividad a(t)')
ax1.set_title('Desactivación del catalizador\npor coque — distintos k_d')
ax1.legend(fontsize=9); ax1.grid(True)

# Panel derecho — X_salida(t) para kd intermedio
kd_mid = kd_vals[1]
a_t_mid = np.exp(-kd_mid * t_s)

# Para cada tiempo, recalcular X de salida con actividad reducida
X_salida = []
for a_val in a_t_mid:
    def pbr_deact(W, state, a=a_val):
        Xv, Tv, yv = state
        Xv = max(min(Xv, 0.9999), 0.0); yv = max(yv, 1e-4)
        k   = k_ref * np.exp(-Ea/R_gas*(1/Tv-1/T_ref)) * a
        CAv = CA0*(1-Xv)/(1+eps*Xv)*yv*(T0/Tv)
        rAv = k * CAv**n_ord
        return [rAv/FA0, (rAv*(-dHrx))/(FA0*rho_cp/CA0), -alpha/(2*yv)*(T0/Tv)]
    try:
        sol2 = solve_ivp(pbr_deact, [0, W_fix], [0, T0, 1.0],
                         t_eval=[W_fix], method='RK45', rtol=1e-6, atol=1e-8)
        X_salida.append(float(sol2.y[0,-1]))
    except:
        X_salida.append(np.nan)

X_salida = np.array(X_salida)
ax2.plot(t_days, X_salida*100, color='#FFD080', lw=2.5)
ax2.axhline(X_des*100, color='#FF4444', ls='--', lw=1.5, label=f'X_des = {X_des*100:.1f}%')
ax2.set_xlabel('Tiempo de operación (días)')
ax2.set_ylabel('X de salida (%)')
ax2.set_title(f'Conversión de salida vs tiempo\n($k_d$ = {kd_mid:.0e} s⁻¹, W = {W_fix/1000:.1f} t catalizador)')
ax2.legend(); ax2.grid(True)
ax2.set_ylim([90, 100])

plt.tight_layout()
plt.show()
print(f"\n📊 Resumen de tiempos de vida:")
for kd, tv in t_vida_label:
    print(f"   k_d = {kd:.0e} s⁻¹  →  t_vida = {tv:.1f} días  ({tv/30:.1f} meses)")


---
# 6. Widget interactivo — Diseño paramétrico del PBR

Explore el efecto de los parámetros de diseño sobre:
- El perfil de conversión X(W)
- La caída de presión P(W)
- La temperatura de salida T_sal
- La masa de catalizador necesaria W*


In [ ]:
def disenar_pbr(k_val=0.12, Ea_kJ=105.0, T_C=350.0, X_obj=0.99,
               alpha_val=0.0028, UA_kJ=0.0, dH_kJ=-63.0,
               CA0_val=0.25, v0_val=15.0):

    T_K    = T_C + 273.15
    Ea_J   = Ea_kJ * 1e3
    dH_J   = dH_kJ * 1e3
    UA_J   = UA_kJ * 1e3
    FA0_v  = CA0_val * v0_val

    def odes_w(W, state):
        Xv, Tv, yv = state
        Xv = max(min(Xv, 0.9999), 0.0); yv = max(yv, 1e-4)
        k   = k_val * np.exp(-Ea_J/R_gas*(1/Tv - 1/T_K))
        CAv = CA0_val*(1-Xv)*yv*(T_K/Tv)
        rAv = k * CAv
        dXdW = rAv / FA0_v
        dTdW = (UA_J*(T_K-Tv) + rAv*(-dH_J)) / (FA0_v * rho_cp / CA0_val)
        dydW = -alpha_val/(2*yv)*(T_K/Tv)
        return [dXdW, dTdW, dydW]

    W_max = 8000
    try:
        sol = solve_ivp(odes_w, [0, W_max], [0.0, T_K, 1.0],
                        t_eval=np.linspace(0, W_max, 2000),
                        method='RK45', rtol=1e-7, atol=1e-9)
        W_v = sol.t; Xv = sol.y[0]; Tv = sol.y[1]; yv = sol.y[2]
    except:
        print("Error en integración — ajustar parámetros"); return

    # W necesario
    try:
        from scipy.interpolate import interp1d
        W_star = float(interp1d(Xv, W_v, fill_value='extrapolate')(X_obj))
    except:
        W_star = np.nan

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#0A0F1E')

    # X(W)
    axes[0].plot(W_v/1000, Xv*100, color='#FFD080', lw=2.5)
    axes[0].axhline(X_obj*100, color='#FF4444', ls='--', lw=1.5, label=f'X_obj={X_obj*100:.1f}%')
    if not np.isnan(W_star) and 0 < W_star < W_max:
        axes[0].axvline(W_star/1000, color='#FF8080', ls=':', lw=1.5)
        axes[0].text(W_star/1000*0.95, X_obj*45, f'W*={W_star/1000:.2f} t',
                     color='#FF8080', ha='right', fontsize=9)
    axes[0].set_xlabel('W (toneladas)'); axes[0].set_ylabel('X (%)')
    axes[0].set_title('Conversión X(W)'); axes[0].legend(); axes[0].grid(True)

    # T(W)
    axes[1].plot(W_v/1000, Tv-273.15, color='#FF8C42', lw=2.5)
    axes[1].set_xlabel('W (toneladas)'); axes[1].set_ylabel('T (°C)')
    axes[1].set_title('Temperatura T(W)'); axes[1].grid(True)

    # P(W)
    axes[2].plot(W_v/1000, yv*100, color='#4A9EFF', lw=2.5, label='P/P₀ (%)')
    axes[2].fill_between(W_v/1000, yv*100, 100, alpha=0.12, color='#4A9EFF')
    axes[2].set_xlabel('W (toneladas)'); axes[2].set_ylabel('P/P₀ (%)')
    axes[2].set_title('Caída de Presión — Ergun'); axes[2].legend(); axes[2].grid(True)

    plt.suptitle(f'PBR Diseño — k={k_val:.3f} dm³/(kg·s), Ea={Ea_kJ:.0f} kJ/mol, T₀={T_C:.0f}°C',
                 color='#C8D8F0', fontweight='bold')
    plt.tight_layout(); plt.show()

    dp = (1 - yv[-1]) * 100
    print(f"{'━'*55}")
    print(f"  W* para X={X_obj*100:.1f}%  : {W_star/1000:.3f} toneladas")
    print(f"  T salida              : {Tv[-1]-273.15:.1f} °C")
    print(f"  Caída de presión      : {dp:.1f}% (P_sal = {yv[-1]*P0:.1f} atm)")
    print(f"  k a T_entrada         : {k_val:.4f} dm³/(kg·s)")
    print(f"{'━'*55}")

interact(disenar_pbr,
    k_val   = widgets.FloatSlider(min=0.01, max=0.5,  step=0.01, value=0.12, description="k [dm³/(kg·s)]", style={'description_width':'140px'}),
    Ea_kJ   = widgets.FloatSlider(min=40,   max=180,  step=5,    value=105,  description="Ea (kJ/mol)",    style={'description_width':'140px'}),
    T_C     = widgets.FloatSlider(min=280,  max=430,  step=5,    value=350,  description="T₀ (°C)",        style={'description_width':'140px'}),
    X_obj   = widgets.FloatSlider(min=0.80, max=0.999,step=0.005,value=0.997,description="X objetivo",     style={'description_width':'140px'}),
    alpha_val=widgets.FloatSlider(min=5e-4, max=8e-3, step=1e-4, value=0.0028,description="α (kg⁻¹)",     style={'description_width':'140px'}),
    UA_kJ   = widgets.FloatSlider(min=0,    max=5,    step=0.1,  value=0.0,  description="UA (kJ/(kg·K))",style={'description_width':'140px'}),
    dH_kJ   = widgets.FloatSlider(min=-200, max=-10,  step=5,    value=-63,  description="ΔH_rx (kJ/mol)",style={'description_width':'140px'}),
    CA0_val = widgets.FloatSlider(min=0.05, max=0.8,  step=0.05, value=0.25, description="C_A0 (mol/dm³)",style={'description_width':'140px'}),
    v0_val  = widgets.FloatSlider(min=5,    max=50,   step=1,    value=15,   description="v₀ (dm³/s)",    style={'description_width':'140px'}),
)


---
# 7. Comparación cuantitativa: PFR homogéneo vs PBR catalítico

| Aspecto | PFR (homogéneo) | PBR (catalítico) |
|---------|----------------|-----------------|
| Variable independiente | Volumen $V$ [dm³] | Masa catalizador $W$ [kg] |
| Ecuación de diseño | $F_{A0}dX/dV = -r_A$ | $F_{A0}dX/dW = -r'_A$ |
| Unidades de velocidad | mol/(dm³·s) | mol/(kg_cat·s) |
| Presión | Constante (homogéneo) | Cae con Ergun (lecho empacado) |
| Temperatura | ODE si no isotérmico | ODE (igual que PFR) |
| Cinética | Ley de potencias | LHHW (mecanismo de superficie) |
| Escala industrial | Columnas de destilación reactiva | Reactores de lecho fijo en refinerías |
| Desactivación | No aplica | Envenenamiento, coque, sinterización |
| Ejemplos industriales | Steam cracking, nitraciones | HDS, reforming catalítico, síntesis NH₃, FTS |

## 7.1 Cuando el PBR supera al PFR homogéneo

El PBR catalítico puede operar a conversiones **mucho más altas** con volúmenes de reactor **mucho menores** porque:
1. $k'$ en superficie catalítica >> $k$ en fase homogénea (superficie activa enorme: $\sim$ 200 m²/g para Al₂O₃)
2. La temperatura puede ser más baja (catalizador baja la energía de activación efectiva)
3. La selectividad puede ser mayor (mecanismo específico sobre la superficie)


---
# 🎓 Resumen del Módulo PBR

| Concepto | Resultado Central |
|----------|------------------|
| Ecuación de diseño | $W = F_{A0}\int_0^X dX/(-r'_A)$ — ODE en $W$, no en $V$ |
| Ergun | $dy/dW = -\alpha/(2y)\cdot(F_T/F_{T0})/(T/T_0)$ — reduce $C_A$ |
| LHHW | $-r'_A = k\,C_A\,C_B/(1+K_AC_A+K_BC_B)^2$ — mecanismo de superficie |
| Caída de presión | Siempre reduce la conversión; $\alpha$ debe calcularse en el diseño |
| Desactivación | $a(t)=e^{-k_dt}$; tiempo de vida = $-\ln(a_\min)/k_d$ |
| PBR vs PFR | Misma forma matemática; diferente unidad de $W$ vs $V$; Ergun añade la ODE de presión |

**Aplicaciones industriales clave:**
- Hidrodesulfuración (HDS): 500+ plantas en el mundo, catalizador CoMo/Al₂O₃
- Reforma catalítica: producción de aromáticos y H₂, catalizador Pt-Re/Al₂O₃  
- Síntesis de amoníaco (Haber-Bosch): catalizador Fe/K₂O, 180 Mt/año
- Fischer-Tropsch (GTL): catalizador Co o Fe, producción de combustibles líquidos

---
*Notebook 5 — PBR · Programa 740484 · Maestría Ingeniería Química · Universidad del Valle · 2024*
